# 02 – Construire la table commune × jour

**Objectif :** fabriquer le tableau d'entraînement : **une ligne par commune et par jour** (2016-2025), avec la cible `feu` = 1 s'il y a eu au moins un départ de feu, 0 sinon.

## Leçon – Pourquoi fabriquer les jours « sans feu » ?

La BDIFF ne liste que les jours **avec** feu. Un modèle qui ne verrait que ces jours-là ne pourrait rien apprendre : il lui faut des exemples des **deux** cas pour comparer (*« qu'est-ce qui distingue un jour avec feu d'un jour sans ? »*).

On va donc :
1. récupérer la liste **complète** des communes ;
2. fabriquer **toutes** les combinaisons commune × jour ;
3. marquer d'un 1 celles où il y a eu un feu.

## Étape 1 – Les feux des Bouches-du-Rhône

In [1]:
import pandas as pd
import requests

df = pd.read_csv(
    "../data/raw/Incendies.csv", sep=";", skiprows=3, encoding="utf-8",
    dtype={"Département": str, "Code INSEE": str},
)
df13 = df[df["Département"] == "13"].copy()
df13["jour"] = pd.to_datetime(df13["Date de première alerte"]).dt.normalize()
print(df13.shape)

(2699, 25)


## Étape 2 – La liste de toutes les communes

Le CSV ne contient que les **112 communes qui ont brûlé**. Les 7 autres doivent aussi apparaître (avec que des 0), sinon le modèle croirait que toutes les communes brûlent. On interroge l'API officielle **geo.api.gouv.fr**, qui renvoie aussi le **centre** de chaque commune (utile pour la météo).

La réponse est du JSON ; `pd.DataFrame(...)` la transforme en tableau. La colonne `centre` contient un petit dictionnaire `{"type": "Point", "coordinates": [lon, lat]}` : une `lambda` (petite fonction jetable) en extrait la longitude (indice 0) et la latitude (indice 1).

In [2]:
url = "https://geo.api.gouv.fr/departements/13/communes?fields=nom,code,centre"
communes = pd.DataFrame(requests.get(url).json())
communes["lon"] = communes["centre"].apply(lambda c: c["coordinates"][0])
communes["lat"] = communes["centre"].apply(lambda c: c["coordinates"][1])
communes = communes[["code", "nom", "lat", "lon"]]
print(len(communes), "communes")
communes.head()

119 communes


,code,nom,lat,lon
0,13001,Aix-en-Provence,43.5360,5.3879
1,13002,Allauch,43.3522,5.5103
2,13003,Alleins,43.7098,5.1558
3,13004,Arles,43.5441,4.6513
4,13005,Aubagne,43.2904,5.5643


## Étape 3 – La grille commune × jour

`how="cross"` fait le **produit cartésien** : chaque commune est associée à chaque jour.

In [3]:
jours = pd.DataFrame({"jour": pd.date_range("2016-01-01", "2025-12-31", freq="D")})
grille = communes[["code"]].merge(jours, how="cross")
print(grille.shape)   # attendu : (434707, 2)

(434707, 2)


**119 communes × 3 653 jours = 434 707 lignes** (3 653 et pas 3 650 : 2016, 2020 et 2024 sont bissextiles).

## Étape 4 – La cible

- `drop_duplicates()` : plusieurs feux le même jour dans la même commune ne font qu'un seul 1 ;
- `how="left"` : on garde **toutes** les lignes de la grille ; celles sans feu reçoivent `NaN`, que `fillna(0)` transforme en 0.

In [4]:
feux = df13[["Code INSEE", "jour"]].drop_duplicates()
feux["feu"] = 1

table = grille.merge(feux, left_on=["code", "jour"], right_on=["Code INSEE", "jour"], how="left")
table["feu"] = table["feu"].fillna(0).astype(int)
table = table.drop(columns="Code INSEE")

print("jours avec feu :", table["feu"].sum())         # attendu : 2480
print(f"taux de feu    : {table['feu'].mean():.2%}")  # attendu : ≈ 0,57 %

jours avec feu : 2480
taux de feu    : 0.57%


**Vérification :** on retrouve bien les 2 480 couples du notebook 01. Si on avait obtenu 0, la jointure aurait échoué (codes INSEE lus comme des nombres d'un côté et du texte de l'autre).

### Le chiffre à retenir : 0,57 %
Les classes sont très **déséquilibrées** : un modèle qui répond toujours « pas de feu » aurait 99,4 % de bonnes réponses… et ne servirait à rien. C'est pour ça qu'on mesurera la qualité avec la **PR-AUC** (notebook 05), pas avec le pourcentage de bonnes réponses.

## Étape 5 – Sauvegarder

In [5]:
communes.to_csv("../data/processed/communes_13.csv", index=False)
table.to_csv("../data/processed/table_13.csv", index=False)

## Récap

- Table de **434 707 lignes** (119 communes × 3 653 jours), cible `feu` avec **2 480 positifs (0,57 %)**.
- Les communes viennent de l'API officielle, avec leur centre (lat/lon) pour la météo.

**Prochaine étape (03) :** télécharger la météo journalière.